# Aula 8 – Análise de Sentimentos - Exemplo 1

**Objetivo:** apresentar o pipeline mínimo com foco em **clareza conceitual** (não performance):

- subjetividade/polaridade (exemplos)
- abordagem léxica (dicionário)
- abordagem supervisionada (Bag-of-Words + Naive Bayes)  
- limitações (negação e sinais conflitantes)


## 1. Preparação

In [ ]:
import numpy as np

## 2. Abordagem léxica (dicionário simples)

Usaremos um léxico **artificial**, apenas para fins didáticos.

In [ ]:
lexico_sentimento = {
    "bom": 1,
    "excelente": 2,
    "ótimo": 2,
    "ruim": -1,
    "péssimo": -2
}

### 2.1 Função de pontuação

In [ ]:
def sentimento_lexico(texto, lexico):
    score = 0
    for palavra in texto.lower().split():
        palavra = palavra.strip(".,!?:;\\\"'()[]{}")
        if palavra in lexico:
            score += lexico[palavra]
    return score

### 2.2 Aplicando o léxico

Interpretação:
- Valores positivos indicam sentimento positivo
- Valores negativos indicam sentimento negativo

In [ ]:
comentarios = [
    "O produto é excelente",
    "O serviço é ruim",
    "O atendimento é ótimo",
    "O sistema é péssimo"
]
for frase in comentarios:
    print(f"{frase:25s} -> score={sentimento_lexico(frase, lexico_sentimento)}")

O produto é excelente     -> score=2
O serviço é ruim          -> score=-1
O atendimento é ótimo     -> score=2
O sistema é péssimo       -> score=-2


## 3. Limitação da abordagem léxica: negação

Observe que a simples presença de palavras positivas/negativas não garante a interpretação correta
quando há **negação** (escopo e estrutura importam).


In [ ]:
comentarios_negacao = [
    "O produto é bom",
    "O produto não é bom",
    "O produto não é ruim"
]

for frase in comentarios_negacao:
    print(f"{frase:25s} -> score={sentimento_lexico(frase, lexico_sentimento)}")

O produto é bom           -> score=1
O produto não é bom       -> score=1
O produto não é ruim      -> score=-1


## 4. Classificação supervisionada (Bag-of-Words + Naive Bayes)

Aqui o modelo aprende associações estatísticas **palavra → classe**.

In [ ]:
textos = [
    "O produto é ótimo",
    "O serviço é péssimo",
    "Gostei do atendimento",
    "Não gostei do sistema",
    "A experiência foi excelente",
    "A entrega foi ruim"
]
# rótulos
sentimentos = [1, 0, 1, 0, 1, 0]  # 1 = positivo, 0 = negativo

### 4.1 Vetorização Bag-of-Words

Como entender o código a seguir:
- `vocabulary_`: Mostra quais palavras foram aprendidas e o índice de cada uma.
- `X.toarray()`:
  - cada linha representa um texto (`textos[i]`);
  - cada coluna é uma palavra do vocabulário ('atendimento' 'do' 'entrega'...);
  - os valores indicam frequência.
- `get_feature_names_out()`: Permite interpretar corretamente cada coluna da matriz. Ex.:  Para o texto  
 *"O produto é ótimo"*  
  temos [0 0 0 0 0 0 0 0 *1* 0 0 0 0 *1*] com valores *1* nos índices 8 e 13, que corresponde às palavras 'produto' e 'ótimo', respectivamente.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
X = vectorizer.fit_transform(textos)
y = sentimentos

# 1. Vocabulário aprendido pelo modelo
print("Vocabulário (palavra -> índice):")
print(vectorizer.vocabulary_)
print()

# 2. Matriz de características (forma esparsa)
print("Matriz Bag-of-Words (forma esparsa):")
print(X)

# 3. Matriz em formato legível (array)
print("Matriz Bag-of-Words (array):")
print(X.toarray())
print()

# 4. Associação entre colunas e palavras
print("Ordem das palavras (colunas da matriz):")
print(vectorizer.get_feature_names_out())

Vocabulário (palavra -> índice):
{'produto': 8, 'ótimo': 13, 'serviço': 11, 'péssimo': 9, 'gostei': 6, 'do': 1, 'atendimento': 0, 'não': 7, 'sistema': 12, 'experiência': 4, 'foi': 5, 'excelente': 3, 'entrega': 2, 'ruim': 10}

Matriz Bag-of-Words (forma esparsa):
<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 17 stored elements and shape (6, 14)>
  Coords	Values
  (0, 8)	1
  (0, 13)	1
  (1, 11)	1
  (1, 9)	1
  (2, 6)	1
  (2, 1)	1
  (2, 0)	1
  (3, 6)	1
  (3, 1)	1
  (3, 7)	1
  (3, 12)	1
  (4, 4)	1
  (4, 5)	1
  (4, 3)	1
  (5, 5)	1
  (5, 2)	1
  (5, 10)	1
Matriz Bag-of-Words (array):
[[0 0 0 0 0 0 0 0 1 0 0 0 0 1]
 [0 0 0 0 0 0 0 0 0 1 0 1 0 0]
 [1 1 0 0 0 0 1 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 1 1 0 0 0 0 1 0]
 [0 0 0 1 1 1 0 0 0 0 0 0 0 0]
 [0 0 1 0 0 1 0 0 0 0 1 0 0 0]]

Ordem das palavras (colunas da matriz):
['atendimento' 'do' 'entrega' 'excelente' 'experiência' 'foi' 'gostei'
 'não' 'produto' 'péssimo' 'ruim' 'serviço' 'sistema' 'ótimo']


### 4.2 Treinamento do Naive Bayes

Classes aprendidas:

- 0 → negativo
- 1 → positivo

In [ ]:
from sklearn.naive_bayes import MultinomialNB

modelo = MultinomialNB()
modelo.fit(X, y)

print("Classes:", modelo.classes_)
print("Priors (em probabilidade):", np.exp(modelo.class_log_prior_))

Classes: [0 1]
Priors (em probabilidade): [0.5 0.5]


### 4.3 Inferência explicada

Vamos exibir:
- vetor da frase
- probabilidades por classe
- classe prevista

In [ ]:
def explicar_predicao(frase):
    Xf = vectorizer.transform([frase])
    print("Frase:", frase)
    print("Vetor:", Xf.toarray())
    print("Probabilidades:", modelo.predict_proba(Xf))
    print("Classe prevista:", modelo.predict(Xf))
    print("-"*60)

explicar_predicao("O produto é excelente")
explicar_predicao("O produto é ruim")
explicar_predicao("O produto é ruim e péssimo")

Frase: O produto é excelente
Vetor: [[0 0 0 1 0 0 0 0 1 0 0 0 0 0]]
Probabilidades: [[0.18615385 0.81384615]]
Classe prevista: [1]
------------------------------------------------------------
Frase: O produto é ruim
Vetor: [[0 0 0 0 0 0 0 0 1 0 1 0 0 0]]
Probabilidades: [[0.47778875 0.52221125]]
Classe prevista: [1]
------------------------------------------------------------
Frase: O produto é ruim e péssimo
Vetor: [[0 0 0 0 0 0 0 0 1 1 1 0 0 0]]
Probabilidades: [[0.63640439 0.36359561]]
Classe prevista: [0]
------------------------------------------------------------


## 5. Discussão

Com poucos dados, termos neutros (ex.: *produto*) podem virar sinal indevido.
Isso acontece com os termos **produto** e **ruim** na frase *"O produto é ruim"*. O classificador combina as probabilidades:

- P(negativo) = 0.4778
- P(positivo) = 0.5222

Já na frase *"O produto é ruim e péssimo"* temos as palavras **ruim** e **péssimo**, que fazem o classificador obter:

- P(negativo) = 0.6364
- P(positivo) = 0.3635

**Exercício -** Aumentar o corpus

Os passos anteriores são repetidos no conjunto de frases a seguir. Veja que a classificação de sentimentos ficou correta.

In [ ]:
textos = [
    # positivos
    "O produto é ótimo",
    "O produto é excelente",
    "Gostei muito do produto",
    "A experiência foi excelente",
    "Atendimento rápido e ótimo",
    "Serviço muito bom",
    "Entrega foi rápida e ótima",
    "Estou satisfeito com o atendimento",
    "O sistema funciona muito bem",
    "Valeu a pena comprar",

    # negativos
    "O produto é ruim",
    "O produto é péssimo",
    "Não gostei do produto",
    "A experiência foi ruim",
    "Atendimento lento e péssimo",
    "Serviço muito ruim",
    "A entrega foi atrasada e ruim",
    "Estou insatisfeito com o atendimento",
    "O sistema falha e é ruim",
    "Não valeu a pena comprar",
]
# rótulos
sentimentos = np.array([1]*10 + [0]*10)
print("N:", len(textos), "| positivo:", int(sentimentos.sum()), "| negativo:", int((1-sentimentos).sum()))

N: 20 | positivo: 10 | negativo: 10


In [ ]:
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(textos)
y = sentimentos

modelo = MultinomialNB()
modelo.fit(X, y)

explicar_predicao("O produto é excelente")
explicar_predicao("O produto é ruim")
explicar_predicao("O produto é ruim e péssimo")

Frase: O produto é excelente
Vetor: [[0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0]]
Probabilidades: [[0.25 0.75]]
Classe prevista: [1]
------------------------------------------------------------
Frase: O produto é ruim
Vetor: [[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0]]
Probabilidades: [[0.85714286 0.14285714]]
Classe prevista: [0]
------------------------------------------------------------
Frase: O produto é ruim e péssimo
Vetor: [[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 1 0 0 0 0 0 0 0 0]]
Probabilidades: [[0.94736842 0.05263158]]
Classe prevista: [0]
------------------------------------------------------------
